# 🧠 Lasso Regression (L1 Regularization) and Feature Selection

Welcome to the hands-on explanation notebook for **Lasso Regression**! In this notebook, we will:
1. Generate a synthetic dataset based on the bounding box classification case study (3 predictive features, 3 pure noise features).
2. Fit Ordinary Least Squares (OLS) Linear Regression to see how it assigns non-zero weights to noisy features.
3. Fit Lasso Regression (L1 Regularization) to observe how it drives noisy feature weights to **exactly zero** (automatic feature selection).
4. Compare it with Ridge Regression (L2 Regularization) which shrinks weights but keeps them non-zero.
5. Implement **Lasso Regression from scratch** using the **Coordinate Descent** algorithm and the **Soft-Thresholding Operator**:
   $$w_j \leftarrow \text{soft\_threshold}(\rho_j, \lambda) / z_j$$

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# Set seed for reproducibility
np.random.seed(42)

## 1. Case Study Data Generation

We generate a bounding box dataset representing:
1.  `width` (True Weight = 1.5)
2.  `height` (True Weight = 0.8)
3.  `aspect_ratio` (True Weight = -1.2)
4.  `pixel_intensity_mean` (Noisy, True Weight = 0.0)
5.  `center_x` (Noisy, True Weight = 0.0)
6.  `center_y` (Noisy, True Weight = 0.0)

Target $y = 1.5 \cdot \text{width} + 0.8 \cdot \text{height} - 1.2 \cdot \text{aspect\_ratio} + \epsilon$

In [ ]:
m = 100 # Number of samples

# Generate predictive features
width = np.random.rand(m, 1) * 4 + 1
height = np.random.rand(m, 1) * 4 + 1
aspect_ratio = width / height

# Generate noisy features
pixel_intensity = np.random.randn(m, 1) * 10 + 128
center_x = np.random.rand(m, 1) * 640
center_y = np.random.rand(m, 1) * 480

# Combine features into matrix X
X = np.hstack((width, height, aspect_ratio, pixel_intensity, center_x, center_y))
feature_names = ['width', 'height', 'aspect_ratio', 'pixel_intensity', 'center_x', 'center_y']

# True linear relationship
true_weights = np.array([1.5, 0.8, -1.2, 0.0, 0.0, 0.0])
noise = np.random.randn(m) * 0.2

# Generate y
y = X @ true_weights + noise

## 2. Comparing OLS, Ridge, and Lasso (Scikit-Learn)

Let's fit all three regression models to see how they handle the noise features.

In [ ]:
# OLS
ols = LinearRegression()
ols.fit(X, y)

# Ridge (L2 penalty)
ridge = Ridge(alpha=10.0)
ridge.fit(X, y)

# Lasso (L1 penalty)
lasso = Lasso(alpha=0.5)
lasso.fit(X, y)

# Compile coefficients into a table
df_coefs = pd.DataFrame({
    'Feature': feature_names,
    'True Weight': true_weights,
    'OLS Learned': ols.coef_,
    'Ridge Learned (L2)': ridge.coef_,
    'Lasso Learned (L1)': lasso.coef_
})

print(df_coefs.to_string(index=False))

Observe how:
-   **OLS** assigns non-zero values (e.g., `-0.0002` or `0.0003`) to noise features (`pixel_intensity`, `center_x`, `center_y`).
-   **Ridge** shrinks the coefficients but leaves them non-zero.
-   **Lasso** sets the noise feature coefficients to **exactly 0.0**!

## 3. Lasso from Scratch (Coordinate Descent)

Since the absolute value function in the L1 penalty is non-differentiable at $w=0$, we cannot use standard gradient descent or the Normal Equation.
Instead, we use **Coordinate Descent**. 
For each feature $j$:
1.  Compute the residual target without feature $j$:
    $$r_i = y_i - \sum_{k \ne j} w_k x_{ik} - b$$
2.  Compute the projection coefficient $\rho_j$:
    $$\rho_j = \sum_{i=1}^m x_{ij} r_i$$
3.  Compute the normalizing factor $z_j$:
    $$z_j = \sum_{i=1}^m x_{ij}^2$$
4.  Update $w_j$ using the **Soft-Thresholding Operator**:
    $$w_j = \text{soft\_threshold}(\rho_j, \lambda) / z_j$$
    where:
    $$\text{soft\_threshold}(\rho, \lambda) = \text{sign}(\rho) \max(0, |\rho| - \lambda)$$

Let's implement this algorithm in python.

In [ ]:
def soft_threshold(rho, lmbda):
    """
    Apply soft thresholding operator.
    """
    if rho > lmbda:
        return rho - lmbda
    elif rho < -lmbda:
        return rho + lmbda
    else:
        return 0.0

def fit_lasso_coordinate_descent(X, y, lmbda, epochs=200):
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0
    
    for epoch in range(epochs):
        # Update bias intercept b (not penalized)
        b = np.mean(y - X @ w)
        
        # Iteratively update each coefficient w_j
        for j in range(n):
            # Compute prediction excluding feature j
            w_except_j = w.copy()
            w_except_j[j] = 0.0
            y_pred_except_j = X @ w_except_j + b
            
            # Compute residual
            r = y - y_pred_except_j
            
            # Compute projection rho_j and normalizer z_j
            rho_j = np.sum(X[:, j] * r)
            z_j = np.sum(X[:, j] ** 2)
            
            # Update weight with soft thresholding
            w[j] = soft_threshold(rho_j, lmbda) / z_j
            
    return w, b

# Run coordinate descent
lambda_param = 50.0  # Equivalent to alpha in sklearn (scaled by samples)
w_scratch, b_scratch = fit_lasso_coordinate_descent(X, y, lambda_param, epochs=300)

print("Lasso coefficients from Scratch Coordinate Descent:")
for name, weight in zip(feature_names, w_scratch):
    print(f"{name:16s}: {weight:.4f}")
print("Bias/Intercept  :", b_scratch)

## 4. Visualizing Feature Selection Path (Lasso Path)

Let's plot how the coefficients change as we increase the regularization parameter $\lambda$. As $\lambda$ increases, features drop to zero one-by-one.

In [ ]:
alphas_path = np.logspace(-2, 2, 100)
coefs = []

for alpha in alphas_path:
    # Scale alpha appropriately for comparison
    w_p, _ = fit_lasso_coordinate_descent(X, y, alpha * m, epochs=100)
    coefs.append(w_p)

coefs = np.array(coefs)

# Plot coefficient values
plt.figure(figsize=(10, 6))
for i in range(len(feature_names)):
    plt.plot(alphas_path, coefs[:, i], label=feature_names[i], linewidth=2)

plt.xscale('log')
plt.xlabel('Regularization Strength (Alpha)')
plt.ylabel('Coefficients')
plt.title('Lasso Path: How Noise Features Drop to Exactly 0 first')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

Observe that the noise features (`center_x`, `center_y`, `pixel_intensity`) instantly drop to 0 even for very small regularization strengths, while the true features (`aspect_ratio`, `width`, `height`) persist much longer!

## 💡 Connection to Deep Learning & YOLO
*   **Model Compression & Sparsity:** L1 regularization is widely used to create **sparse neural networks**. By adding L1 penalties to weights, we can force many weights to become exactly zero. Pruning algorithms then remove these zero-weight connections entirely, reducing the model size (fewer parameters) and speeding up inference, which is crucial for running YOLO models in real-time on edge devices.